In [10]:
from pathlib import Path
from dataclasses import dataclass, replace
import dataclasses
import json
import re

import kdrag.solvers.tla as tla
from hypothesis import given, settings, strategies as st

Connect python functions to TLA first? Hypothesis testing?

It could be kind of fun to mock locking or something to emit action labels / trace events. This could be somewhat transparent (opaque? depends on your preferred term)


In [4]:
%%file /tmp/HourClock.tla
---- MODULE HourClock ----
EXTENDS Naturals

VARIABLE hr

HCini == hr \in 1 .. 12
HCnxt == hr' = IF hr = 12 THEN 1 ELSE hr + 1
Next == HCnxt \/ UNCHANGED hr
HC == HCini /\ [][HCnxt]_hr
====

Overwriting /tmp/HourClock.tla


In [5]:
%%file /tmp/HourClock.cfg
INIT HCini
NEXT Next

Overwriting /tmp/HourClock.cfg


In [7]:
@dataclass(frozen=True)
class ClockState:
    hr: int


def tick(state):
    return replace(state, hr=state.hr % 13 + 1)


def stutter(state):
    return replace(state)


actions = {"tick": tick, "stutter": stutter}


@st.composite
def hourclock_traces(draw, max_steps=10):
    state = ClockState(draw(st.integers(1, 12)))
    trace = [state]
    names = draw(st.lists(st.sampled_from(list(actions)), max_size=max_steps))
    for name in names:
        state = actions[name](state)
        trace.append(state)
    return trace

In [13]:
%%prun
def validate_trace(trace : list[ClockState]) -> bool:
    states = [dataclasses.asdict(state) for state in trace]
    data = {
        "vars": ["hr"],
        "counterexample": {
            "state": [[i, state] for i, state in enumerate(states, 1)],
            "action": [],
        },
    }
    tracefile = "/tmp/hourclock_trace.json"
    Path(tracefile).write_text(json.dumps(data))
    out = tla.run_tools([
        "tlc2.TLC",
        "-workers", "1",
        "-loadTrace", "json", tracefile,
        "-config", "/tmp/HourClock",
        "/tmp/HourClock.tla",
    ]).decode()
    print(out)
    #assert "No error has been found." in out, "Trace is invalid"
    depth = int(re.search(r"The depth .* is (\d+)\.", out).group(1))
    return depth >= len(trace)


assert validate_trace([ClockState(12), ClockState(1), ClockState(1)])
assert not validate_trace([ClockState(1), ClockState(3)])

TLC2 Version 2026.07.14.071606 (rev: 227f61b)
(Use the -nowarning option to disable this warning.)
Running breadth-first search Model-Checking with fp 103 and seed 1047999301453507687 with 1 worker on 16 cores with 15207MB heap and 64MB offheap memory [pid: 1334020] (Linux 7.0.0-28-generic amd64, Ubuntu 21.0.11 64bit, MSBDiskFPSet, DiskStateQueue).
Parsing file /tmp/HourClock.tla
Parsing file /tmp/tlc-8557857232466255485/Naturals.tla (jar:file:/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/tla2tools.jar!/tla2sany/StandardModules/Naturals.tla)
Parsing file /tmp/tlc-8557857232466255485/_TLCTrace.tla (jar:file:/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/tla2tools.jar!/tla2sany/StandardModules/_TLCTrace.tla)
Parsing file /tmp/tlc-8557857232466255485/_JsonTrace.tla (jar:file:/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/tla2tools.jar!/tla2sany/StandardModules/_JsonTrace.tla)
Parsing file /tmp/tlc-8557857232466255485/TLC.t

         3569 function calls (3551 primitive calls) in 1.515 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       78    1.469    0.019    1.469    0.019 {method 'poll' of 'select.poll' objects}
        4    0.034    0.008    0.034    0.008 {method 'poll' of 'select.epoll' objects}
        2    0.002    0.001    0.002    0.001 {method '__exit__' of 'sqlite3.Connection' objects}
       78    0.002    0.000    1.471    0.019 selectors.py:402(select)
        2    0.001    0.001    1.309    0.654 subprocess.py:2062(_communicate)
        2    0.001    0.000    0.001    0.000 {built-in method _posixsubprocess.fork_exec}
       82    0.001    0.000    0.001    0.000 {built-in method posix.read}
        2    0.001    0.000    0.001    0.000 {built-in method posix.waitpid}
       23    0.000    0.000    0.000    0.000 socket.py:623(send)
        2    0.000    0.000    0.002    0.001 subprocess.py:807(__init__)
       82    0.000   

In [9]:
@settings(max_examples=10, deadline=None)
@given(hourclock_traces())
def test_python_hourclock_refines_tla(trace):
    assert validate_trace(trace), trace


test_python_hourclock_refines_tla()

AssertionError: [ClockState(hr=9), ClockState(hr=10), ClockState(hr=11), ClockState(hr=12), ClockState(hr=13)]

cerberus
cbmc

renode
actually control the hardware? gdb scrpit from python

https://arxiv.org/abs/2404.16075 merz Validating Traces of Distributed Programs Against TLA+ Specifications

https://www.youtube.com/watch?v=NZmON-XmrkI Validating System Executions with the TLA+ Tools Markus A Kuppe, Microsoft

https://www.youtube.com/watch?v=W6DrQk8o5tk

https://docs.tlapl.us/using:tlc:trace_validation 

https://pron.github.io/files/Trace.pdf ron pressler trace vliation 2018

It's surprising there is an json tlc module. also an IO module?

tla importer could wrap exprssion in dummy module.
```python
def expr(e : str, variables=[], constants=[]):
    with write() as f:
        f.write("----- KDRAGDUMMY --------)
        f.write(f"VARIABLES {v})
        f.write(f"KDRAGEXPR == {e}\n")
        f.write(f"==================")
    mod = Module.load_file("/tmp/KDRAGDUMMY.tla")
    mod.infer_sorts()
    return mod.action("KDRAGEXPR")
```

Yea, maybe I'm getting closer to SPIN?

cocotb might be kind of interesting...
spike or sail derived emulator?
Try a bunch of them?




In [39]:
%%file /tmp/hour.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>   

typedef struct ClockState {
    int hr;
} ClockState;

ClockState state;

void tick(){
    state.hr = state.hr % 13 + 1;
}

void main(){
    srand(time(NULL));
    state.hr = rand() % 12 + 1;
    printf("[");
    for(int t = 0; t < 100; t++){
        printf("[%d, { hr : %d }]\n", t, state.hr);
        tick();
    }
    printf("]");
}


Overwriting /tmp/hour.c


In [40]:
! gcc -o /tmp/hour /tmp/hour.c && /tmp/hour

[[0, { hr : 3 }]
[1, { hr : 4 }]
[2, { hr : 5 }]
[3, { hr : 6 }]
[4, { hr : 7 }]
[5, { hr : 8 }]
[6, { hr : 9 }]
[7, { hr : 10 }]
[8, { hr : 11 }]
[9, { hr : 12 }]
[10, { hr : 13 }]
[11, { hr : 1 }]
[12, { hr : 2 }]
[13, { hr : 3 }]
[14, { hr : 4 }]
[15, { hr : 5 }]
[16, { hr : 6 }]
[17, { hr : 7 }]
[18, { hr : 8 }]
[19, { hr : 9 }]
[20, { hr : 10 }]
[21, { hr : 11 }]
[22, { hr : 12 }]
[23, { hr : 13 }]
[24, { hr : 1 }]
[25, { hr : 2 }]
[26, { hr : 3 }]
[27, { hr : 4 }]
[28, { hr : 5 }]
[29, { hr : 6 }]
[30, { hr : 7 }]
[31, { hr : 8 }]
[32, { hr : 9 }]
[33, { hr : 10 }]
[34, { hr : 11 }]
[35, { hr : 12 }]
[36, { hr : 13 }]
[37, { hr : 1 }]
[38, { hr : 2 }]
[39, { hr : 3 }]
[40, { hr : 4 }]
[41, { hr : 5 }]
[42, { hr : 6 }]
[43, { hr : 7 }]
[44, { hr : 8 }]
[45, { hr : 9 }]
[46, { hr : 10 }]
[47, { hr : 11 }]
[48, { hr : 12 }]
[49, { hr : 13 }]
[50, { hr : 1 }]
[51, { hr : 2 }]
[52, { hr : 3 }]
[53, { hr : 4 }]
[54, { hr : 5 }]
[55, { hr : 6 }]
[56, { hr : 7 }]
[57, { hr : 8 }]
[58, { 

In [48]:
%%file /tmp/hour.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
  
typedef struct ClockState {
    int hr;
} ClockState;

ClockState state;

void tick(){
    state.hr = state.hr % 13 + 1;
}

int main(){
    srand(time(NULL));
    state.hr = rand() % 12 + 1;
    for(int t = 0; t < 100; t++){
        tick();
    }
    return 0;
}

Overwriting /tmp/hour.c


In [49]:
! gcc -g -Wall -o /tmp/hour /tmp/hour.c

import gdb only works inside GDB's embedded Python.
 GDB/MI i  https://sourceware.org/gdb/current/onlinedocs/gdb.html/GDB_002fMI.html
 Is this overwrought?
 Should I just make a python scriper and load it from inside gdb or script gdb in some other way


 https://www.youtube.com/watch?v=xt9v5t4_zvE lisa roach - extended gdb with python. Very fun
Could this be a road into some whackasmackadoo tower of interpreters stuff?


Hmm. Control renode via gdb? https://renode.readthedocs.io/en/latest/debugging/gdb.html
https://github.com/matgla/Renode_RP2040
https://github.com/wokwi/rp2040js


Worry: instrumentation may change system. printf has locks in pico for example
Make instrumentation so cheap you leave it on? (antithesis right?)
Or further testing required anyhow


In [ ]:
import os
os.getpid()

In [ ]:
%%file /tmp/printhello.py

import gdb
gdb.execute("call \"Python)



In [ ]:
%%file /tmp/hourclock.gdb
set pagination off
set confirm off
set debuginfod enabled off
break tick
commands
  silent
  printf "CLOCK %d\n", state.hr
  continue
end
run

In [ ]:
import subprocess

result = subprocess.run(
    ["gdb", "-q", "--batch", "-x", "/tmp/hourclock.gdb", "/tmp/hour"],
    capture_output=True, text=True, check=True,
)
c_trace = [
    ClockState(int(hr))
    for hr in re.findall(r"^CLOCK (\d+)$", result.stdout, re.MULTILINE)
]
c_trace[:10], len(c_trace)

In [58]:
import sys
sys.version
sys.executable


'/home/philip/philzook58.github.io/.venv/bin/python'

1285037

In [56]:
! gdb -ex "python import sys; print(sys.version); print(sys.executable)" -ex "quit"

GNU gdb (Ubuntu 15.1-1ubuntu1~24.04.1) 15.1
Copyright (C) 2024 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
Type "show copying" and "show warranty" for details.
This GDB was configured as "x86_64-linux-gnu".
Type "show configuration" for configuration details.
For bug reporting instructions, please see:
<https://www.gnu.org/software/gdb/bugs/>.
Find the GDB manual and other documentation resources online at:
    <http://www.gnu.org/software/gdb/documentation/>.

For help, type "help".
Type "apropos word" to search for commands related to "word".
3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
/usr/bin/python


# Renode


In [61]:
%%file /tmp/hourclock_rv.c
typedef struct { volatile unsigned int hr; } ClockState;
volatile ClockState state = {1};
extern char __stack_top[];
int main(void);

__attribute__((naked, section(".text.start")))
void _start(void) {
    __asm__ volatile("la sp, __stack_top\n"
                     "call main\n"
                     "ebreak\n"
                     "1: j 1b");
}

__attribute__((noinline)) void tick(void) { state.hr = state.hr % 12 + 1; }
__attribute__((noinline)) void tick_done(void) {}

int main(void) {
    for(int i = 0; i < 10; i++) {
        tick();
        tick_done();
    }
    return 0;
}


Overwriting /tmp/hourclock_rv.c


In [62]:
%%file /tmp/hourclock_rv.ld
ENTRY(_start)
SECTIONS {
    . = 0x80000000;
    .text : { KEEP(*(.text.start)) *(.text*) }
    .rodata : { *(.rodata*) }
    .data : { *(.data*) }
    .bss : { *(.bss*) *(COMMON) }
    . = ALIGN(16);
    . += 0x1000;
    __stack_top = .;
}


Overwriting /tmp/hourclock_rv.ld


In [2]:
%%file /tmp/hourclock.resc
mach create "hourclock"
machine LoadPlatformDescriptionFromString """
cpu: CPU.RiscV64 @ sysbus
    cpuType: "rv64imac"
    privilegedArchitecture: PrivilegedArchitecture.Priv1_12
    timeProvider: empty

ram: Memory.MappedMemory @ sysbus 0x80000000
    size: 0x100000
"""
sysbus LoadELF @/tmp/hourclock_rv.elf
machine StartGdbServer 3333


Overwriting /tmp/hourclock.resc


In [64]:
%%file /tmp/hourclock_renode.py
import gdb
import json

trace = []

class TickDone(gdb.Breakpoint):
    def stop(self):
        trace.append({"hr": int(gdb.parse_and_eval("state.hr"))})
        return False

TickDone("tick_done")
gdb.execute("monitor start")
gdb.execute("continue")
print("CLOCKTRACE " + json.dumps(trace))


Overwriting /tmp/hourclock_renode.py


In [65]:
!riscv64-unknown-elf-gcc -march=rv64imac -mabi=lp64 -mcmodel=medany \
    -g -O0 -ffreestanding -nostdlib -Wl,-T,/tmp/hourclock_rv.ld \
    -o /tmp/hourclock_rv.elf /tmp/hourclock_rv.c


/usr/lib/gcc/riscv64-unknown-elf/13.2.0/../../../riscv64-unknown-elf/bin/ld: warning: /tmp/hourclock_rv.elf has a LOAD segment with RWX permissions


In [3]:
import json
import re
import subprocess

renode = subprocess.Popen(
    ["dotnet", "/opt/renode/bin/Renode.dll",
     "--disable-xwt", "--plain", "--config",
     "/tmp/hourclock-renode-config", "/tmp/hourclock.resc"],
    stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True,
)
try:
    result = subprocess.run([
        "gdb-multiarch", "-q", "--batch", "/tmp/hourclock_rv.elf",
        "-ex", "target remote :3333",
        "-ex", "source /tmp/hourclock_renode.py",
    ], capture_output=True, text=True, timeout=20)
    result.check_returncode()
    match = re.search(r"^CLOCKTRACE (.*)$", result.stdout, re.MULTILINE)
    assert match, result.stdout + result.stderr
    renode_trace = [ClockState(**st) for st in json.loads(match.group(1))]
finally:
    renode.terminate()
    renode.wait(timeout=5)

renode_trace


TimeoutExpired: Command '['gdb-multiarch', '-q', '--batch', '/tmp/hourclock_rv.elf', '-ex', 'target remote :3333', '-ex', 'source /tmp/hourclock_renode.py']' timed out after 20 seconds

# Pico trace

interrupt triggering?
send over hypothesis generated interrupt schedule?
hardware watchpoints

Could ingest TLA spec and directly look for violations in the fuzzer.
python script could generate hypotheses itself and just stream out traces.

If we ingest TLA spec, instead of using TLC, could check directly in python. Best to do both? Maybe being in python could inform hypothesis more (in interview they mentioned it peeks at source code?)

```
import tla
tla.Module.of_()


```

Errors in gdb script

TICKSTART TICKEND. Maybe actions are kind of spread over time not an instant?












In [ ]:
Path("/tmp/picohour").mkdir(exist_ok=True)

In [7]:
%%file /tmp/picohour/CMakeLists.txt
cmake_minimum_required(VERSION 3.13)
set(PICO_BOARD pico2)
set(PICO_SDK_PATH /home/philip/.pico-sdk/sdk/2.3.0)
set(PICO_TOOLCHAIN_PATH /home/philip/.pico-sdk/toolchain/15_2_Rel1)
set(picotool_DIR /home/philip/.pico-sdk/picotool/2.3.0/picotool)
include(/home/philip/.pico-sdk/sdk/2.3.0/external/pico_sdk_import.cmake)

project(hourclock C CXX ASM)
pico_sdk_init()

add_executable(hourclock hourclock.c)
target_link_libraries(hourclock pico_stdlib)

Overwriting /tmp/picohour/CMakeLists.txt


In [8]:
%%file /tmp/picohour/hourclock.c
#include "pico/stdlib.h"

typedef struct { volatile unsigned int hr; } ClockState;
volatile ClockState state = {1};

__attribute__((noinline)) void trace_point(void) { __asm volatile ("nop"); }
__attribute__((noinline)) void trace_done(void) { __asm volatile ("nop"); }

int main(void) {
    trace_point();
    for (int i = 0; i < 10; i++) {
        state.hr = state.hr % 12 + 1;
        trace_point();
    }
    trace_done();
    while (true) tight_loop_contents();
}


Overwriting /tmp/picohour/hourclock.c


In [11]:
import subprocess
subprocess.run(["cmake", "-S", "/tmp/picohour", "-B", "/tmp/picohour/build",
                "-G", "Ninja", "-DCMAKE_BUILD_TYPE=Debug"], check=True)
subprocess.run(["cmake", "--build", "/tmp/picohour/build"], check=True)

PICO_SDK_PATH is /home/philip/.pico-sdk/sdk/2.3.0
Target board (PICO_BOARD) is 'pico2'.
Using board configuration from /home/philip/.pico-sdk/sdk/2.3.0/src/boards/include/boards/pico2.h
Pico Platform (PICO_PLATFORM) is 'rp2350-arm-s'.


-- The C compiler identification is GNU 13.2.1
-- The CXX compiler identification is GNU 13.2.1
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/arm-none-eabi-gcc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/arm-none-eabi-gcc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/arm-none-eabi-g++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done


Build type is Debug
Using regular optimized debug build (set PICO_DEOPTIMIZED_DEBUG=1 to de-optimize)
Using picotool from /home/philip/.pico-sdk/picotool/2.3.0/picotool/picotool


-- Found Python3: /home/philip/philzook58.github.io/.venv/bin/python3.12 (found version "3.12.3") found components: Interpreter
-- Configuring done (0.7s)


TinyUSB available at /home/philip/.pico-sdk/sdk/2.3.0/lib/tinyusb/hw/bsp/rp2040; enabling build support for USB.
Compiling TinyUSB with CFG_TUSB_DEBUG=1
BTstack available at /home/philip/.pico-sdk/sdk/2.3.0/lib/btstack
cyw43-driver available at /home/philip/.pico-sdk/sdk/2.3.0/lib/cyw43-driver
mbedtls available at /home/philip/.pico-sdk/sdk/2.3.0/lib/mbedtls
lwIP available at /home/philip/.pico-sdk/sdk/2.3.0/lib/lwip
C library type is newlib


-- Generating done (0.1s)
-- Build files have been written to: /tmp/picohour/build
[1/4] Generating bs2_default_padded.S
[2/4] Building ASM object pico-sdk/src/rp2350/boot_stage2/CMakeFiles/bs2_default_library.dir/bs2_default_padded.S.o
[3/4] Building C object CMakeFiles/hourclock.dir/hourclock.c.o
[4/4] Linking CXX executable hourclock.elf


CompletedProcess(args=['cmake', '--build', '/tmp/picohour/build'], returncode=0)

In [12]:
%%file /tmp/picohour/trace.py

import gdb
import json

trace = []

class TracePoint(gdb.Breakpoint):
    def stop(self):
        trace.append({"hr": int(gdb.parse_and_eval("state.hr"))})
        return False

TracePoint("trace_point")
gdb.Breakpoint("trace_done")
gdb.execute("monitor reset init")
gdb.execute("load")
gdb.execute("continue")
print("CLOCKTRACE " + json.dumps(trace))


Overwriting /tmp/picohour/trace.py


In [16]:
import re
import json
from dataclasses import dataclass, replace
@dataclass
class ClockState:
    hr : int
openocd = subprocess.Popen([
    "/home/philip/.pico-sdk/openocd/0.12.0+dev/openocd",
    "-s", "/home/philip/.pico-sdk/openocd/0.12.0+dev/scripts",
    "-f", "interface/cmsis-dap.cfg", "-f", "target/rp2350.cfg",
    "-c", "adapter speed 5000",
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in openocd.stdout:
        if "Listening on port 3333 for gdb connections" in line:
            break
    result = subprocess.run([
        "gdb-multiarch", "-q", "--batch", "/tmp/picohour/build/hourclock.elf",
        "-ex", "target extended-remote localhost:3333",
        "-ex", "source /tmp/picohour/trace.py",
    ], capture_output=True, text=True, check=True, timeout=30)
finally:
    openocd.terminate()
    openocd.wait(timeout=5)

match = re.search(r"^CLOCKTRACE (.*)$", result.stdout, re.MULTILINE)
assert match, result.stdout + result.stderr
pico_trace = [ClockState(**st) for st in json.loads(match.group(1))]
pico_trace

[ClockState(hr=1),
 ClockState(hr=2),
 ClockState(hr=3),
 ClockState(hr=4),
 ClockState(hr=5),
 ClockState(hr=6),
 ClockState(hr=7),
 ClockState(hr=8),
 ClockState(hr=9),
 ClockState(hr=10),
 ClockState(hr=11)]

# rust



In [12]:
%%file /tmp/hourclock.rs

struct ClockState {
    hr: u32,
}
impl ClockState {
    fn new(hr: u32) -> Self {
        assert!(hr >= 1 && hr <= 12);
        ClockState { hr }
    }
    fn tick(&mut self) {
        self.hr = (self.hr + 1) % 12;
    }
}

fn main(){
    let mut clock = ClockState::new(12);
    for i in 0..100 {
        clock.tick();
        'mylabel: for _ in 0..0 {}
        // MYLABEL
        println!("CLOCK {} {{ hr : {} }}", i, clock.hr);
    }
}


Overwriting /tmp/hourclock.rs


Ok so add some sigil and grep for it.
It breaks at the next one?
Or add ranges that correspond to actions?

Tick = ("main:23","main:24")

check for atomiciity? If another action starts
watchpoint on all variables. But what if variable is in 

// IncStart
gIndex++
// IncEnd

Then we probably would find a discrepancy


Ok. the other bit is random stepping


In [ ]:
with open("/tmp/hourclock.rs", "r") as f:
    for n, line in enumerate(f.readlines()):
        if "MYLABEL" in line:
            print(n+1) # line labels start at 1

20


https://www.sourceware.org/gdb/current/onlinedocs/gdb.html/Tracepoints.html#Tracepoints This sounds really useful

ActionStart/ActionEnd?
Or a single Action time point?
Labelling can be embedded via comments
I haven’t really found a good way to have stable rust labels
Loop labels i don’t think persist in debug data?
Python can grep for comments and hence line number though
I think the diffference of next vs step would be enough for the simple race to be picked up as gindex++ not being atomic
If i used start end style action labels, it’ll see that increment isn’t atomic

In [13]:
! rustc -g -C debuginfo=2 -o /tmp/hourclock /tmp/hourclock.rs && /tmp/hourclock

  --> /tmp/hourclock.rs:19:9
   |
19 |         'mylabel: for _ in 0..0 {}
   |         ^^^^^^^^
   |
   = note: `#[warn(unused_labels)]` (part of `#[warn(unused)]`) on by default




CLOCK 0 { hr : 1 }
CLOCK 1 { hr : 2 }
CLOCK 2 { hr : 3 }
CLOCK 3 { hr : 4 }
CLOCK 4 { hr : 5 }
CLOCK 5 { hr : 6 }
CLOCK 6 { hr : 7 }
CLOCK 7 { hr : 8 }
CLOCK 8 { hr : 9 }
CLOCK 9 { hr : 10 }
CLOCK 10 { hr : 11 }
CLOCK 11 { hr : 0 }
CLOCK 12 { hr : 1 }
CLOCK 13 { hr : 2 }
CLOCK 14 { hr : 3 }
CLOCK 15 { hr : 4 }
CLOCK 16 { hr : 5 }
CLOCK 17 { hr : 6 }
CLOCK 18 { hr : 7 }
CLOCK 19 { hr : 8 }
CLOCK 20 { hr : 9 }
CLOCK 21 { hr : 10 }
CLOCK 22 { hr : 11 }
CLOCK 23 { hr : 0 }
CLOCK 24 { hr : 1 }
CLOCK 25 { hr : 2 }
CLOCK 26 { hr : 3 }
CLOCK 27 { hr : 4 }
CLOCK 28 { hr : 5 }
CLOCK 29 { hr : 6 }
CLOCK 30 { hr : 7 }
CLOCK 31 { hr : 8 }
CLOCK 32 { hr : 9 }
CLOCK 33 { hr : 10 }
CLOCK 34 { hr : 11 }
CLOCK 35 { hr : 0 }
CLOCK 36 { hr : 1 }
CLOCK 37 { hr : 2 }
CLOCK 38 { hr : 3 }
CLOCK 39 { hr : 4 }
CLOCK 40 { hr : 5 }
CLOCK 41 { hr : 6 }
CLOCK 42 { hr : 7 }
CLOCK 43 { hr : 8 }
CLOCK 44 { hr : 9 }
CLOCK 45 { hr : 10 }
CLOCK 46 { hr : 11 }
CLOCK 47 { hr : 0 }
CLOCK 48 { hr : 1 }
CLOCK 49 { hr : 2 }
C

watch clock.hr
condition 1 
display clock.hr
list .



In [22]:
%%file /tmp/hourclock_gdb.py
import gdb
#print("hello world")


gdb.write("hello world\n")

gdb.execute("set pagination off")
gdb.execute("file /tmp/hourclock")
gdb.execute("info functions")
#gdb.execute("list Clockstate.tick")

# shell commands
res = gdb.execute("! ls")
print(res) # nothin. Ok
gdb.execute("set confirm off")
gdb.execute("set debuginfod enabled off")
#gdb.execute("start")
gdb.execute("start")
import random

gdb.execute(f"n {random.randint(1, 12)}")
gdb.execute("run") # run > /tmp/myoutfile


gdb.execute("quit")


Overwriting /tmp/hourclock_gdb.py


In [23]:
! gdb -ex "source /tmp/hourclock_gdb.py"    # /tmp/hourclock -ex "quit"

GNU gdb (Ubuntu 15.1-1ubuntu1~24.04.1) 15.1
Copyright (C) 2024 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.
Type "show copying" and "show warranty" for details.
This GDB was configured as "x86_64-linux-gnu".
Type "show configuration" for configuration details.
For bug reporting instructions, please see:
<https://www.gnu.org/software/gdb/bugs/>.
Find the GDB manual and other documentation resources online at:
    <http://www.gnu.org/software/gdb/documentation/>.

For help, type "help".
Type "apropos word" to search for commands related to "word".
hello world
of file /tmp/hourclock.
Use `info auto-load python-scripts [REGEXP]' to list them.
All defined functions:

File /home/philip/.rustup/toolchains/stable-x86_64-unknown-linux-gnu/lib/rustlib/src/rust/library/core/src/fmt/mod.rs:
729:	static fn core::fmt:

In [24]:
! rustup target list

aarch64-apple-darwin
aarch64-apple-ios
aarch64-apple-ios-macabi
aarch64-apple-ios-sim
aarch64-apple-tvos
aarch64-apple-tvos-sim
aarch64-apple-visionos
aarch64-apple-visionos-sim
aarch64-apple-watchos
aarch64-apple-watchos-sim
aarch64-linux-android
aarch64-pc-windows-gnullvm
aarch64-pc-windows-msvc
aarch64-unknown-freebsd
aarch64-unknown-fuchsia
aarch64-unknown-linux-gnu
aarch64-unknown-linux-musl
aarch64-unknown-linux-ohos
aarch64-unknown-none
aarch64-unknown-none-softfloat
aarch64-unknown-uefi
arm-linux-androideabi
arm-unknown-linux-gnueabi
arm-unknown-linux-gnueabihf
arm-unknown-linux-musleabi
arm-unknown-linux-musleabihf
arm64ec-pc-windows-msvc
armv5te-unknown-linux-gnueabi
armv5te-unknown-linux-musleabi
armv7-linux-androideabi
armv7-unknown-linux-gnueabi
armv7-unknown-linux-gnueabihf
armv7-unknown-linux-musleabi
armv7-unknown-linux-musleabihf
armv7-unknown-linux-ohos
armv7a-none-eabi
armv7a-none-eabihf
armv7r-none-eabi
armv7r-none-eabihf
armv8r-none-eabihf
i586-unknown-linux-gnu
i5

# Qemu
https://qemu-project.gitlab.io/qemu/system/gdb.html
ok, but qemu-system and userland are different beasts


In [41]:
%%file /tmp/hourclock.rs

struct ClockState {
    hr: u32,
}
impl ClockState {
    fn new(hr: u32) -> Self {
        assert!(hr >= 1 && hr <= 12);
        ClockState { hr }
    }
    fn tick(&mut self) {
        self.hr = (self.hr + 1) % 12;
    }
}

fn main(){
    let mut clock = ClockState::new(12);
    loop {
        clock.tick();
    }
}

Overwriting /tmp/hourclock.rs


In [42]:
! rustc -g -C opt-level=0 /tmp/hourclock.rs -o /tmp/hourclock-x86_64

In [51]:
%%file /tmp/gdb_qemu.py
import gdb
import json
import os

gdb.execute("set debuginfod enabled off")
gdb.execute("target remote :1234")
gdb.Breakpoint("hourclock::ClockState::tick")
gdb.execute("continue")
results = []
for n in json.loads(os.environ["GDB_STEPS"]):
    gdb.execute(f"next {n}")
    results.append(int(gdb.parse_and_eval("self.hr")))
gdb.write("GDB_RESULTS " + json.dumps(results) + "\n")


Overwriting /tmp/gdb_qemu.py


In [52]:
import json, os, subprocess, time
from hypothesis import given, settings, strategies as st

def run_gdb(steps):
    qemu = subprocess.Popen(["qemu-x86_64", "-g", "1234", "/tmp/hourclock-x86_64"])
    try:
        time.sleep(0.1)
        gdb = subprocess.run([
            "gdb", "-q", "--batch", "/tmp/hourclock-x86_64",
            "-ex", "source /tmp/gdb_qemu.py",
        ], capture_output=True, text=True, env=os.environ | {"GDB_STEPS": json.dumps(steps)})
    finally:
        qemu.terminate()
        qemu.wait()
    assert gdb.returncode == 0, gdb.stdout + gdb.stderr
    marker = "GDB_RESULTS "
    line = next((line for line in gdb.stdout.splitlines() if line.startswith(marker)), None)
    assert line is not None, gdb.stdout + gdb.stderr
    return json.loads(line.removeprefix(marker))

@settings(max_examples=10, deadline=None)
@given(st.lists(st.integers(min_value=1, max_value=12), min_size=1, max_size=10))
def test(steps):
    assert len(run_gdb(steps)) == len(steps)

test()

[{'steps': [1], 'results': [1]},
 {'steps': [3], 'results': [1]},
 {'steps': [3, 7, 4], 'results': [1, 2, 3]},
 {'steps': [7, 7, 2, 10, 7, 4, 8, 6, 4, 9],
  'results': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]},
 {'steps': [11, 1, 11, 8, 11, 10, 8, 6], 'results': [1, 2, 2, 3, 4, 5, 6, 7]},
 {'steps': [10, 9], 'results': [1, 2]},
 {'steps': [5, 6, 7, 11, 1, 3, 2, 1, 9, 12],
  'results': [1, 2, 3, 4, 5, 5, 6, 7, 7, 8]},
 {'steps': [8, 1, 11, 7, 12, 11, 4, 8, 6, 5],
  'results': [1, 2, 2, 3, 4, 5, 6, 7, 8, 9]},
 {'steps': [2, 6, 8, 6, 10, 11, 4, 7, 8, 9],
  'results': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]},
 {'steps': [2], 'results': [1]}]

# riscv32 system

In [86]:
%%file /tmp/hourclock_rv32.rs
#![no_std]
#![no_main]

use core::arch::{asm, global_asm};
use core::panic::PanicInfo;

global_asm!(r#"
    .section .text.init
    .globl _start
_start:
    la sp, _stack_top
    call rust_main
1:  j 1b
"#);

#[repr(C)]
pub struct ClockState { pub hr: u32 }

#[no_mangle]
pub static mut state: ClockState = ClockState { hr: 12 };

#[no_mangle]
pub extern "C" fn rust_main() -> ! {
    for _ in 0..100 {
        unsafe { state.hr = state.hr % 13 + 1 }
    }
    loop { unsafe { asm!("wfi") } }
}

#[panic_handler]
fn panic(_: &PanicInfo) -> ! { loop {} }

Overwriting /tmp/hourclock_rv32.rs


In [87]:
%%file /tmp/rv32.ld
ENTRY(_start)
SECTIONS {
    . = 0x80000000;
    .text : { KEEP(*(.text.init)) *(.text*) }
    .rodata : { *(.rodata*) }
    .data : { *(.data*) }
    .bss (NOLOAD) : {
        *(.bss*)
        . = ALIGN(16);
        . += 4K;
        _stack_top = .;
    }
}

Overwriting /tmp/rv32.ld


In [88]:
! rustc +1.94.0 --target riscv32imac-unknown-none-elf -g -C opt-level=0 -C panic=abort -C link-arg=-T/tmp/rv32.ld -C link-arg=--no-relax /tmp/hourclock_rv32.rs -o /tmp/hourclock-rv32.elf

In [89]:
%%file /tmp/gdb_rv32.py
import gdb, json, site, subprocess, time

site.addsitedir("/home/philip/philzook58.github.io/.venv/lib/python3.12/site-packages")
from hypothesis import given, settings, strategies as st

elf = gdb.current_progspace().filename
qemu = subprocess.Popen([
    "qemu-system-riscv32", "-machine", "virt", "-bios", "none",
    "-kernel", elf, "-S", "-gdb", "tcp::1235",
    "-display", "none", "-serial", "none", "-monitor", "none",
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    time.sleep(0.1)
    gdb.execute("set suppress-cli-notifications on")
    gdb.execute("set architecture riscv:rv32", to_string=True)
    gdb.execute("target remote :1235", to_string=True)
    try:
        gdb.execute("set language rust", to_string=True)
        gdb.parse_and_eval("hourclock_rv32::state.hr")
        state = "hourclock_rv32::state.hr"
    except gdb.error:  # the C example
        gdb.execute("set language c", to_string=True)
        state = "state.hr"
    watch = gdb.Breakpoint(
        state, type=gdb.BP_WATCHPOINT,
        wp_class=gdb.WP_WRITE, internal=True)
    watch.silent = True
    runs = []

    @settings(deadline=None)
    @given(st.integers(min_value=1, max_value=10))
    def test(ticks):
        gdb.execute("monitor system_reset", to_string=True)
        gdb.execute("load", to_string=True)
        trace = [int(gdb.parse_and_eval(state))]
        for _ in range(ticks):
            gdb.execute("continue", to_string=True)
            trace.append(int(gdb.parse_and_eval(state)))
        runs.append(trace)

    test()
    for trace in runs:
        print(json.dumps({
            "vars": ["hr"],
            "counterexample": {
                "state": [[i, {"hr": hr}] for i, hr in enumerate(trace, 1)],
                "action": [],
            },
        }))
finally:
    try:
        gdb.execute("disconnect", to_string=True)
    except gdb.error:
        pass
    if qemu.poll() is None:
        qemu.terminate()
    qemu.wait()

Overwriting /tmp/gdb_rv32.py


In [90]:
trace_lines = ! gdb-multiarch -q --batch /tmp/hourclock-rv32.elf -x /tmp/gdb_rv32.py
traces = [json.loads(line) for line in trace_lines]
for trace in traces:
    print(json.dumps(trace))

{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}]], "action": []}}
{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}], [3, {"hr": 1}], [4, {"hr": 2}], [5, {"hr": 3}], [6, {"hr": 4}], [7, {"hr": 5}], [8, {"hr": 6}]], "action": []}}
{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}], [3, {"hr": 1}], [4, {"hr": 2}], [5, {"hr": 3}]], "action": []}}
{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}], [3, {"hr": 1}], [4, {"hr": 2}]], "action": []}}
{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}], [3, {"hr": 1}]], "action": []}}
{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}], [3, {"hr": 1}], [4, {"hr": 2}], [5, {"hr": 3}], [6, {"hr": 4}], [7, {"hr": 5}], [8, {"hr": 6}], [9, {"hr": 7}], [10, {"hr": 8}], [11, {"hr": 9}]], "action": []}}
{"vars": ["hr"], "counterexample": {"state": [[1, {"hr": 12}], [2, {"hr": 13}], [3, {"

In [92]:
from pathlib import Path
import kdrag.solvers.tla as tla
import re
for i, trace in enumerate(traces):
    tracefile = f"/tmp/hourclock_trace_{i}.json"
    print(trace)
    Path(tracefile).write_text(json.dumps(trace))
    out = tla.run_tools([
        "tlc2.TLC",
        "-workers", "1",
        "-loadTrace", "json", tracefile,
        "-config", "/tmp/HourClock",
        "/tmp/HourClock.tla",
    ]).decode()
    depth = int(re.search(r"The depth .* is (\d+)\.", out).group(1))
    if depth < len(trace["counterexample"]["state"]):
        states = trace["counterexample"]["state"]
        raise AssertionError(
            f"TLC rejected transition: {states[depth - 1][1]} -> {states[depth][1]}"
        )
print(f"{len(traces)} traces passed")

{'vars': ['hr'], 'counterexample': {'state': [[1, {'hr': 12}], [2, {'hr': 13}]], 'action': []}}


AssertionError: TLC rejected transition: {'hr': 12} -> {'hr': 13}

## C

In [ ]:
%%file /tmp/hourclock_rv32.c
__asm__(
    ".section .text.init\n"
    ".globl _start\n"
    "_start:\n"
    "la sp, _stack_top\n"
    "call c_main\n"
    "1: j 1b\n"
);

typedef struct { volatile unsigned int hr; } ClockState;
ClockState state = {12};

__attribute__((noreturn)) void c_main(void) {
    for (int i = 0; i < 100; ++i)
        state.hr = state.hr % 12 + 1;
    for (;;) __asm__ volatile("wfi");
}

In [ ]:
! riscv64-unknown-elf-gcc -march=rv32imac -mabi=ilp32 -mcmodel=medany -g -O0 -ffreestanding -nostdlib -T /tmp/rv32.ld -Wl,--no-relax,--no-warn-rwx-segments /tmp/hourclock_rv32.c -o /tmp/hourclock-rv32-c.elf

In [ ]:
! gdb-multiarch -q --batch /tmp/hourclock-rv32-c.elf -x /tmp/gdb_rv32.py

# Simple Interrupt



In [98]:
%%file /tmp/GIndexAtomic.tla
---------------- MODULE GIndexAtomic ----------------
EXTENDS Naturals

VARIABLE gIndex

Init == gIndex = 0
MainStep == /\ gIndex # 0 /\ gIndex' = gIndex - 1
IntStep == gIndex' = gIndex + 1
Next == MainStep \/ IntStep \/ UNCHANGED gIndex
Bound == gIndex <= 10
====================================================

Overwriting /tmp/GIndexAtomic.tla


In [99]:
%%file /tmp/GIndexAtomic.cfg
INIT Init
NEXT Next
CONSTRAINT Bound

Overwriting /tmp/GIndexAtomic.cfg


clint core local interrupt - MSIP = macine software interrupt pending
plic platform level interrupt controller

In [100]:
%%file /tmp/gindex_rv32.c
#include <stdint.h>

#define CLINT_MSIP (*(volatile uint32_t *)0x02000000)

__asm__(
    ".section .text.init\n"
    ".globl _start\n"
    "_start:\n"
    "la sp, _stack_top\n"
    "call main\n"
    "1: j 1b\n"
);

volatile uint32_t gIndex = 0;

void __attribute__((interrupt("machine"), aligned(4))) interrupt_handler(void) {
    CLINT_MSIP = 0;
    gIndex++;
}

void __attribute__((noinline)) main_loop(void) {
    for (;;)
        if (gIndex)
            gIndex--;
}

void main(void) {
    __asm__ volatile("csrw mtvec, %0" :: "r"(interrupt_handler));
    __asm__ volatile("csrsi mie, 8");
    __asm__ volatile("csrsi mstatus, 8");
    main_loop();
}

Overwriting /tmp/gindex_rv32.c


In [101]:
! riscv64-unknown-elf-gcc -march=rv32imac_zicsr -mabi=ilp32 -mcmodel=medany -g -O0 -ffreestanding -nostdlib -T /tmp/rv32.ld -Wl,--no-relax,--no-warn-rwx-segments /tmp/gindex_rv32.c -o /tmp/gindex-rv32.elf

In [95]:
%%file /tmp/gdb_gindex.py
import gdb, json, site, socket, subprocess, time

site.addsitedir("/home/philip/philzook58.github.io/.venv/lib/python3.12/site-packages")
from hypothesis import given, settings, strategies as st

elf = gdb.current_progspace().filename
qemu = subprocess.Popen([
    "qemu-system-riscv32", "-machine", "virt", "-bios", "none",
    "-kernel", elf, "-S", "-gdb", "tcp::1236",
    "-qtest", "tcp:127.0.0.1:1237,server=on,wait=off",
    "-display", "none", "-serial", "none", "-monitor", "none",
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
qtest = None

try:
    time.sleep(0.1)
    gdb.execute("set suppress-cli-notifications on")
    gdb.execute("set architecture riscv:rv32", to_string=True)
    gdb.execute("target remote :1236", to_string=True)
    gdb.execute("maintenance packet Qqemu.sstep=0x5", to_string=True)  # allow IRQs while stepping
    gdb.execute("set language c", to_string=True)
    qtest = socket.create_connection(("127.0.0.1", 1237))
    runs = []

    def irq_enabled():
        mstatus = int(gdb.parse_and_eval("$mstatus"))
        mie = int(gdb.parse_and_eval("$mie"))
        mip = int(gdb.parse_and_eval("$mip"))
        return mstatus & 8 and mie & 8 and not mip & 8

    def interrupt():
        qtest.sendall(b"writel 0x02000000 1\n")
        assert qtest.recv(64).startswith(b"OK")

    @settings(max_examples=10, deadline=None, derandomize=True)
    @given(st.lists(st.integers(1, 30), min_size=5, max_size=10))
    def test(schedule):
        gdb.execute("monitor system_reset", to_string=True)
        gdb.execute("load", to_string=True)
        stop = gdb.Breakpoint("main_loop", temporary=True, internal=True)
        stop.silent = True
        gdb.execute("continue", to_string=True)
        trace = [int(gdb.parse_and_eval("gIndex"))]
        for n in schedule:
            if irq_enabled():
                interrupt()
            for _ in range(n):
                gdb.execute("stepi", to_string=True)
                value = int(gdb.parse_and_eval("gIndex"))
                if value != trace[-1]:
                    trace.extend([trace[-1], value])
        runs.append(trace)

    test()
    for trace in runs:
        print(json.dumps({
            "vars": ["gIndex"],
            "counterexample": {
                "state": [[i, {"gIndex": value}]
                          for i, value in enumerate(trace, 1)],
                "action": [],
            },
        }))
finally:
    if qtest is not None:
        qtest.close()
    try:
        gdb.execute("disconnect", to_string=True)
    except gdb.error:
        pass
    if qemu.poll() is None:
        qemu.terminate()
    qemu.wait()

Overwriting /tmp/gdb_gindex.py


In [96]:
interrupt_trace_lines = ! gdb-multiarch -q --batch /tmp/gindex-rv32.elf -x /tmp/gdb_gindex.py
interrupt_traces = [json.loads(line) for line in interrupt_trace_lines]
for trace in interrupt_traces:
    print(json.dumps(trace))

{"vars": ["gIndex"], "counterexample": {"state": [[1, {"gIndex": 0}], [2, {"gIndex": 1}], [3, {"gIndex": 2}], [4, {"gIndex": 3}], [5, {"gIndex": 4}], [6, {"gIndex": 5}]], "action": []}}
{"vars": ["gIndex"], "counterexample": {"state": [[1, {"gIndex": 0}], [2, {"gIndex": 1}], [3, {"gIndex": 2}], [4, {"gIndex": 3}], [5, {"gIndex": 4}], [6, {"gIndex": 0}]], "action": []}}
{"vars": ["gIndex"], "counterexample": {"state": [[1, {"gIndex": 0}], [2, {"gIndex": 1}], [3, {"gIndex": 2}], [4, {"gIndex": 0}], [5, {"gIndex": 1}], [6, {"gIndex": 2}], [7, {"gIndex": 3}], [8, {"gIndex": 1}], [9, {"gIndex": 2}], [10, {"gIndex": 0}], [11, {"gIndex": 1}], [12, {"gIndex": 2}], [13, {"gIndex": 3}], [14, {"gIndex": 1}], [15, {"gIndex": 2}], [16, {"gIndex": 1}], [17, {"gIndex": 0}]], "action": []}}
{"vars": ["gIndex"], "counterexample": {"state": [[1, {"gIndex": 0}], [2, {"gIndex": 1}], [3, {"gIndex": 0}], [4, {"gIndex": 1}], [5, {"gIndex": 2}], [6, {"gIndex": 0}], [7, {"gIndex": 1}], [8, {"gIndex": 0}], [9, 

In [97]:
for i, trace in enumerate(interrupt_traces):
    tracefile = f"/tmp/gindex_trace_{i}.json"
    Path(tracefile).write_text(json.dumps(trace))
    out = tla.run_tools([
        "tlc2.TLC", "-workers", "1",
        "-loadTrace", "json", tracefile,
        "-config", "/tmp/GIndexAtomic",
        "/tmp/GIndexAtomic.tla",
    ]).decode()
    depth = int(re.search(r"The depth .* is (\d+)\.", out).group(1))
    states = trace["counterexample"]["state"]
    if depth < len(states):
        raise AssertionError(
            f"TLC rejected transition: {states[depth - 1][1]} -> {states[depth][1]}"
        )
print(f"{len(interrupt_traces)} traces passed")

AssertionError: TLC rejected transition: {'gIndex': 4} -> {'gIndex': 0}

## Non-atomic spec

In [ ]:
%%file /tmp/GIndexNonAtomic.tla
--------------- MODULE GIndexNonAtomic ---------------
EXTENDS Naturals

VARIABLE gIndex, mainpc, gindexlocal, received, processed
vars == <<gIndex, mainpc, gindexlocal, received, processed>>

Init ==
    /\ gIndex = 0
    /\ gindexlocal = 0
    /\ mainpc = "load"
    /\ received = 0
    /\ processed = 0

Load ==
    /\ mainpc = "load"
    /\ gIndex # 0
    /\ mainpc' = "dec"
    /\ gindexlocal' = gIndex - 1
    /\ UNCHANGED <<gIndex, received, processed>>

Dec ==
    /\ mainpc = "dec"
    /\ mainpc' = "load"
    /\ gIndex' = gindexlocal
    /\ processed' = processed + 1
    /\ UNCHANGED <<gindexlocal, received>>

IRQInc ==
    /\ gIndex' = gIndex + 1
    /\ received' = received + 1
    /\ UNCHANGED <<mainpc, gindexlocal, processed>>

Next == Load \/ Dec \/ IRQInc \/ UNCHANGED vars
NoLost == received = processed + gIndex
Bound == /\ gIndex <= 10 /\ gindexlocal <= 10
         /\ received <= 10 /\ processed <= 10
======================================================

In [ ]:
%%file /tmp/GIndexNonAtomic.cfg
INIT Init
NEXT Next
CONSTRAINT Bound

In [ ]:
for i, trace in enumerate(interrupt_traces):
    tracefile = f"/tmp/gindex_refined_trace_{i}.json"
    Path(tracefile).write_text(json.dumps(trace))
    out = tla.run_tools([
        "tlc2.TLC", "-workers", "1",
        "-loadTrace", "json", tracefile,
        "-config", "/tmp/GIndexNonAtomic",
        "/tmp/GIndexNonAtomic.tla",
    ]).decode()
    depth = int(re.search(r"The depth .* is (\d+)\.", out).group(1))
    assert depth >= len(trace["counterexample"]["state"]), out
print(f"{len(interrupt_traces)} refined traces passed")

In [ ]:
%%file /tmp/GIndexNonAtomicCheck.cfg
INIT Init
NEXT Next
CONSTRAINT Bound
INVARIANT NoLost

In [ ]:
try:
    tla.run_tools([
        "tlc2.TLC", "-workers", "1",
        "-config", "/tmp/GIndexNonAtomicCheck",
        "/tmp/GIndexNonAtomic.tla",
    ])
except RuntimeError as error:
    assert "Invariant NoLost is violated" in str(error)
    print(error)
else:
    raise AssertionError("expected NoLost violation")

# factoring

Qemu class

class for specific system. How to detect if interrupts are enabled

Yeah, maybe this is overwrought. But I also want guidance about how not to end up with false negatives.


```
class Tracer
    def __init__(self, mod):
        self.vars = []
        self.traces = []
        self.tla_mod = mod
        for action in mod.actions:
            # find action in file comments
    def 

    def add_var(self, tla_name, gdb_expr):
        assert tla_name in self.tla_mod.vars
    def trace(self):
        event = [gdb.parse_and_eval(gdb_expr) for tla_name, gdb_expr in self.vars]
        self.traces[-1].append(event)
    def new_trace(self):
        assert self.vars == self.tla_mod.vars # we know how to monitor all of them.
        self.traces.append([])
```


Moving more out of the python and into tla is probably good, _if_ we can compare tla specs
lean + gdb mi? or python import lean something? Somewhat arcane achitecture. but maybe.

Try it on hardware debug pi pico
write blog post


qtree - config params?
mtree

Single step qemu?
qmp vs hmp

qtest? https://www.qemu.org/docs/master/devel/testing/qtest.html

In [107]:
! echo -e "info mtree\n quit" | qemu-system-riscv32 \
  -machine opentitan \
  -S -display none -monitor stdio

QEMU 8.2.2 monitor - type 'help' for more information
(qemu) iininfinfoinfo info minfo mtinfo mtrinfo mtreinfo mtree
address-space: cpu-memory-0
address-space: memory
  0000000000000000-ffffffffffffffff (prio 0, i/o): system
    0000000000008000-000000000000ffff (prio 0, rom): riscv.lowrisc.ibex.rom
    0000000010000000-000000001001ffff (prio 0, ram): riscv.lowrisc.ibex.ram
    0000000020000000-00000000200fffff (prio 0, rom): riscv.lowrisc.ibex.flash
    0000000040000000-00000000400003ff (prio 0, i/o): ibex-uart
    0000000040040000-000000004004003f (prio -1000, i/o): riscv.lowrisc.ibex.gpio
    0000000040050000-0000000040051fff (prio -1000, i/o): riscv.lowrisc.ibex.spi_device
    0000000040080000-000000004008007f (prio -1000, i/o): riscv.lowrisc.ibex.i2c
    00000000400e0000-00000000400e003f (prio -1000, i/o): riscv.lowrisc.ibex.pattgen
    0000000040100000-00000000401003ff (prio 0, i/o): ibex-timer
    0000000040130000-0000000040131fff (prio -1000, i/o): riscv.lowrisc.ibex.otp_ctrl
 